# MedExtract AI — Full Rebuild

A complete, organized rebuild of the project: data exploration, gold labeling, the full
prompt-engineering journey (V1 → V2 → V3 → FINAL), validation, live medical verification
(Phase 3.5), and evaluation. Run top to bottom. Uses `gemini-3.5-flash-lite` throughout,
with multi-key fallback for rate limits.

**Sections:**
1. Setup
2. Phase 1 — Read & Understand the Dataset
3. Phase 2 — Labeled Evaluation Set
4. Phase 3 — Prompt Iteration (V1 → V2 → V3 → FINAL)
5. Phase 4 — Validation & Repair Loop
6. Phase 3.5 — Live Medical Verification (diagnosis / medication / dosing / ICD-10)
7. Phase 6 — Evaluation


---
## 1. Setup

In [1]:
import os, json, time, re
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai.errors import ClientError
from pydantic import BaseModel
from typing import Optional, List, Literal

load_dotenv()
pd.set_option('display.max_colwidth', None)

api_keys = [os.environ.get(f"GEMINI_API_KEY_{i}") for i in (1, 2, 3)]
api_keys = [k for k in api_keys if k]
if not api_keys and os.environ.get("GEMINI_API_KEY"):
    api_keys = [os.environ["GEMINI_API_KEY"]]
clients = [genai.Client(api_key=k) for k in api_keys]
print(f"Loaded {len(clients)} API key(s)")

MODEL_NAME = "gemini-3.5-flash-lite"
_current_client_index = 0

def generate_with_fallback(**kwargs):
    global _current_client_index
    last_error = None
    for attempt in range(len(clients)):
        idx = (_current_client_index + attempt) % len(clients)
        try:
            response = clients[idx].models.generate_content(**kwargs)
            _current_client_index = idx
            return response
        except ClientError as e:
            if e.code == 429:
                print(f"Key #{idx+1} rate-limited, trying next key...")
                last_error = e
                continue
            raise
    raise last_error

def parse_json_response(raw_text):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip())
    return json.loads(cleaned)

print("Setup complete.")


Loaded 3 API key(s)
Setup complete.


---
## 2. Phase 1 — Read & Understand the Dataset

In [2]:
clinical = pd.read_csv("../data/raw/clinical_notes.csv")
diaries = pd.read_csv("../data/raw/patient_diaries.csv")

print("Clinical notes:", clinical.shape, "| unique texts:", clinical["text"].nunique())
print("Patient diaries:", diaries.shape, "| unique texts:", diaries["text"].nunique())
print()
print("Clinical label distribution:\n", clinical["label"].value_counts())
print()
print("Diary label distribution:\n", diaries["label"].value_counts())


Clinical notes: (1000, 7) | unique texts: 57
Patient diaries: (1000, 7) | unique texts: 64

Clinical label distribution:
 label
depression       605
no depression    395
Name: count, dtype: int64

Diary label distribution:
 label
no depression    514
depression       486
Name: count, dtype: int64


In [3]:
clinical_unique = clinical.drop_duplicates(subset="text")[["text", "label"]].reset_index(drop=True)
diaries_unique = diaries.drop_duplicates(subset="text")[["text", "label"]].reset_index(drop=True)
print(clinical_unique.shape, diaries_unique.shape)


(57, 2) (64, 2)


---
## 3. Phase 2 — Labeled Evaluation Set

**Finding:** the dataset's own `label` column is unreliable — manual review found it only
agrees with careful judgment 69.2% of the time on clinical notes and 41.4% on diary entries
(non-ambiguous rows only). Gold labels below were built by hand, using the note's actual
content rather than trusting the CSV label. See `docs/APPROACH.md` for the full methodology.

In [4]:
clinical_gold_labels = [
    "depression","depression","depression","ambiguous","depression",
    "depression","depression","depression","depression","ambiguous",
    "ambiguous","not depression","depression","ambiguous","depression",
    "depression","ambiguous","ambiguous","depression","depression",
    "ambiguous","depression","depression","ambiguous","depression",
    "ambiguous","not depression","depression","ambiguous","depression",
    "depression","depression","ambiguous","depression","depression",
    "ambiguous","ambiguous","depression","depression","depression",
    "depression","ambiguous","depression","depression","ambiguous",
    "depression","depression","depression","depression","ambiguous",
    "depression","ambiguous","depression","depression","depression",
    "depression","ambiguous"
]
diary_gold_labels = [
    "not depression","ambiguous","depression","ambiguous","ambiguous",
    "ambiguous","ambiguous","ambiguous","not depression","depression",
    "ambiguous","depression","not depression","not depression","depression",
    "ambiguous","ambiguous","ambiguous","depression","depression",
    "ambiguous","ambiguous","not depression","ambiguous","depression",
    "depression","depression","ambiguous","not depression","ambiguous",
    "ambiguous","not depression","not depression","ambiguous","ambiguous",
    "ambiguous","depression","ambiguous","ambiguous","ambiguous",
    "ambiguous","depression","ambiguous","ambiguous","not depression",
    "ambiguous","ambiguous","not depression","ambiguous","not depression",
    "ambiguous","not depression","ambiguous","depression","depression",
    "ambiguous","not depression","ambiguous","not depression","ambiguous",
    "ambiguous","ambiguous","not depression","not depression"
]

assert len(clinical_gold_labels) == len(clinical_unique)
assert len(diary_gold_labels) == len(diaries_unique)
clinical_unique["gold_label"] = clinical_gold_labels
diaries_unique["gold_label"] = diary_gold_labels
print("Gold labels applied.")


Gold labels applied.


In [5]:
clinical_negative_words = {"low", "fatigued", "sad", "overwhelmed", "unstable", "isolated", "anxious"}
clinical_word_map = {
    "low": "low mood", "fatigued": "fatigue", "sad": "sadness",
    "overwhelmed": "feeling overwhelmed", "unstable": "mood instability",
    "isolated": "social isolation", "anxious": "anxiety",
}

def make_clinical_gold(row):
    text = row["text"]; text_lower = text.lower(); gold_label = row["gold_label"]
    symptoms = []; follow_up = None
    if "demonstrated symptoms of depression" in text_lower or "shows signs of depression" in text_lower:
        symptoms.append("depressive symptoms")
        if "demonstrated symptoms" in text_lower:
            follow_up = "further assessment needed"
    for word in clinical_negative_words:
        if re.search(rf"\b{word}\b", text_lower):
            label = clinical_word_map[word]
            if label not in symptoms:
                symptoms.append(label)
    if "referral to a psychiatrist" in text_lower:
        follow_up = "referral to psychiatrist recommended"
    if "lack of interest in activities" in text_lower:
        symptoms.append("anhedonia (lack of interest in activities)")
    if "unable to concentrate" in text_lower:
        symptoms.append("difficulty concentrating")
    if "lacks motivation" in text_lower:
        symptoms.append("low motivation")
    if "needs support" in text_lower and follow_up is None:
        follow_up = "patient needs support"
    diagnosis = gold_label if gold_label in ("depression", "not depression") else None
    return {
        "chief_complaint": None, "symptoms": symptoms if symptoms else None, "diagnosis": diagnosis,
        "medical_history": None, "medications": None, "procedures": None, "follow_up": follow_up,
        "summary": text, "risk_indicators": "isolation" if "isolated" in text_lower else None,
        "urgency": "moderate" if follow_up else "low", "_gold_label_for_eval": gold_label
    }

diary_clause_signals = {
    "struggled to get out of bed and had no motivation": {"symptoms": ["low motivation", "difficulty getting out of bed"], "urgency": "moderate"},
    "felt overwhelmed with sadness and isolation": {"symptoms": ["sadness", "feeling overwhelmed", "social isolation"], "urgency": "moderate", "risk": "isolation"},
    "found it hard to focus today; it was a heavy day": {"symptoms": ["difficulty concentrating", "low mood"], "urgency": "low"},
    "felt a bit off but managed to accomplish some tasks": {"symptoms": ["mild mood disturbance"], "urgency": "low"},
    "wrote in my journal and expressed my thoughts": {"symptoms": None, "urgency": "low"},
    "felt a glimmer of hope today, looking forward to tomorrow": {"symptoms": None, "urgency": "low"},
    "took a walk in the evening, which slightly lifted my spirits": {"symptoms": None, "urgency": "low"},
    "shared a laugh with a friend, which helped me momentarily": {"symptoms": None, "urgency": "low"},
}
opener_negative_words = {"confused", "down", "empty", "overwhelmed"}

def make_diary_gold(row):
    text = row["text"]; gold_label = row["gold_label"]
    opener_match = re.search(r"I felt (\w+) today", text)
    opener_word = opener_match.group(1) if opener_match else None
    clause_info = None
    for clause, info in diary_clause_signals.items():
        if clause in text:
            clause_info = info; break
    symptoms = list(clause_info["symptoms"]) if clause_info and clause_info["symptoms"] else []
    if opener_word in opener_negative_words and opener_word not in [s.lower() for s in symptoms]:
        symptoms.append(f"reports feeling {opener_word}")
    diagnosis = gold_label if gold_label in ("depression", "not depression") else None
    return {
        "chief_complaint": None, "symptoms": symptoms if symptoms else None, "diagnosis": diagnosis,
        "medical_history": None, "medications": None, "procedures": None, "follow_up": None,
        "summary": text, "risk_indicators": clause_info.get("risk") if clause_info else None,
        "urgency": clause_info["urgency"] if clause_info else "low", "_gold_label_for_eval": gold_label
    }

clinical_unique["gold_json"] = clinical_unique.apply(make_clinical_gold, axis=1)
diaries_unique["gold_json"] = diaries_unique.apply(make_diary_gold, axis=1)
clinical_unique["source"] = "clinical"
diaries_unique["source"] = "diary"

eval_set = pd.concat([
    clinical_unique[["source", "text", "label", "gold_label", "gold_json"]],
    diaries_unique[["source", "text", "label", "gold_label", "gold_json"]]
], ignore_index=True)

os.makedirs("../data/labeled", exist_ok=True)
os.makedirs("../data/eval_sets", exist_ok=True)
clinical_unique.to_csv("../data/labeled/clinical_gold_labels.csv", index=False)
diaries_unique.to_csv("../data/labeled/diaries_gold_labels.csv", index=False)
eval_set.to_json("../data/eval_sets/full_eval_set.jsonl", orient="records", lines=True)

print(f"Eval set built: {len(eval_set)} notes")
print(eval_set["gold_label"].value_counts())


Eval set built: 121 notes
gold_label
ambiguous         53
depression        50
not depression    18
Name: count, dtype: int64


---
## 4. Phase 3 — Prompt Iteration (V1 → V2 → V3 → FINAL)

Documented journey. Each version's known issue is noted; FINAL resolves all of them.

### V1 — Baseline (known issue: no format enforcement, conversational responses possible)

In [6]:
V1 = """ I need your help with extracting some information from clinical notes and patient diaries. I will provide you with a text, and I want you to extract the following information,
and if there any filed that is not includes just put null, and also dont just guess or infer:
Return ONLY valid JSON matching this exact schema:
{
  "chief_complaint": string or null,
  "symptoms": array of strings or null,
  "diagnosis": string or null,
  "medical_history": string or null,
  "medications": array of strings or null,
  "procedures": array of strings or null,
  "follow_up": string or null,
  "summary": string,
  "risk_indicators": string or null,
  "urgency": "low" or "moderate" or "high"
}
"""

def extract_v1(note_text):
    response = generate_with_fallback(model=MODEL_NAME, contents=f"{V1}\n\nText:\n{note_text}")
    return response.text

print(extract_v1("Patient has fluctuating moods, feeling low."))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "chief_complaint": "fluctuating moods, feeling low",
  "symptoms": [
    "fluctuating moods",
    "feeling low"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "Patient presents with fluctuating moods and feeling low.",
  "risk_indicators": null,
  "urgency": "moderate"
}


### V2 — Fixed conversational tone, added null instruction (known issue: null vs. empty-array inconsistency, nested medications object)

In [7]:
V2 = """
    You are a data extractor. Extract only information that is explicitly stated in the text, and structure the output in this JSON format:
    {  "chief_complaint": null,
        "symptoms": [],
        "diagnosis": null,
        "medical_history": [],
        "medications": [
            {   "name": null,
                "dose": null,
                "frequency": null,
                "duration": null    }
            ],
        "procedures": [],
        "follow_up": null,
        "summary": null,
        "risk_indicators": [],
        "urgency": null
    }
    If any field is not included just put null, and also don't just guess or infer.
    Output only the JSON object — no explanation, no commentary, no questions, no extra text.
"""

def extract_v2(note_text):
    response = generate_with_fallback(model=MODEL_NAME, contents=f"{V2}\n\nText:\n{note_text}")
    return response.text

print(extract_v2("Patient has fluctuating moods, feeling low."))


{  "chief_complaint": null,
        "symptoms": [
            "fluctuating moods",
            "feeling low"
        ],
        "diagnosis": null,
        "medical_history": [],
        "medications": [],
        "procedures": [],
        "follow_up": null,
        "summary": null,
        "risk_indicators": [],
        "urgency": null
    }


### V3 — Added Therapist persona, stronger null rule (known issue: format block and example contradicted each other)

In [8]:
V3 = """
    Role : You are a Therapist.

    Task : Your task is to extract required information from the clinical notes i give you.

    Requirements :
    - Only use information that was mentioned.
    - Do not invent or infer missing information, even if it seems likely.
    - If a field is not mentioned, use null for that field. Do not use empty strings or empty arrays for missing data — use null consistently.
    - Use the exact field names and structure provided in the output format below.
    - Output ONLY the JSON object. Do not include any explanation, commentary, questions, or markdown formatting.

    Output Format :
    {  "chief_complaint": null,
            "symptoms": [],
            "diagnosis": null,
            "medical_history": [],
            "medications": [
                {   "name": null,
                    "dose": null,
                    "frequency": null,
                    "duration": null    }
                ],
            "procedures": [],
            "follow_up": null,
            "summary": null,
            "risk_indicators": [],
            "urgency": null
    }
    as in JSON object
"""

def extract_v3(note_text):
    response = generate_with_fallback(model=MODEL_NAME, contents=f"{V3}\n\nText:\n{note_text}")
    return response.text

print(extract_v3("Patient has fluctuating moods, feeling low."))


{  "chief_complaint": null,
        "symptoms": [
            "fluctuating moods",
            "feeling low"
        ],
        "diagnosis": null,
        "medical_history": [],
        "medications": [
            {   "name": null,
                "dose": null,
                "frequency": null,
                "duration": null    }
            ],
        "procedures": [],
        "follow_up": null,
        "summary": null,
        "risk_indicators": [],
        "urgency": null
}


### FINAL — Consistent null rule (no contradiction with example), few-shot example, always-populate summary/urgency

In [9]:
FINAL = """
    Role: You are a Therapist extracting structured data from clinical notes.

    Rules:
    - Only extract what is explicitly stated. Never infer or guess.
    - Always fill in "summary" with a one-sentence summary, even if nothing else is found.
    - Always set "urgency" to "low", "moderate", or "high" — never leave it blank.
    - Use null for any field with no information. Do not use empty arrays or placeholder objects.

    The output must be a valid JSON object with exactly these fields:
    {  "chief_complaint": null,
            "symptoms": null,
            "diagnosis": null,
            "medical_history": null,
            "medications": null,
            "procedures": null,
            "follow_up": null,
            "summary": null,
            "risk_indicators": null,
            "urgency": null
        }

    Example:
    Note: "Patient feels sad; a referral to a psychiatrist may be necessary."
    Output:
    {
    "chief_complaint": null,
    "symptoms": ["sadness"],
    "diagnosis": null,
    "medical_history": null,
    "medications": null,
    "procedures": null,
    "follow_up": "referral to psychiatrist recommended",
    "summary": "Patient reports sadness; psychiatric referral suggested.",
    "risk_indicators": null,
    "urgency": "moderate"
    }

    Now extract from this note, following the exact same format:
"""

def extract_final(note_text):
    response = generate_with_fallback(model=MODEL_NAME, contents=f"{FINAL}\n\nText:\n{note_text}")
    return response.text

print(extract_final("Patient has fluctuating moods, feeling low."))


{
  "chief_complaint": null,
  "symptoms": [
    "fluctuating moods",
    "feeling low"
  ],
  "diagnosis": null,
  "medical_history": null,
  "medications": null,
  "procedures": null,
  "follow_up": null,
  "summary": "Patient reports fluctuating moods and feeling low.",
  "risk_indicators": null,
  "urgency": "low"
}


### Full-batch run of FINAL across all 121 notes (produces the real Phase 3 results)

In [10]:
def run_full_batch(eval_df, extract_fn, delay=4):
    results = []
    for i, row in eval_df.iterrows():
        try:
            raw_output = extract_fn(row["text"])
            parsed = parse_json_response(raw_output)
            success = True
        except Exception as e:
            parsed = None
            success = False
            print(f"FAILED at row {i}: {e}")
        results.append({
            "index": i, "source": row["source"], "text": row["text"],
            "gold_label": row["gold_label"], "gold_json": row["gold_json"],
            "model_output": parsed, "parse_success": success
        })
        time.sleep(delay)
        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(eval_df)}")
    return results

# Uncomment to run the full batch (uses ~121 API calls):
batch_results = run_full_batch(eval_set, extract_final, delay=4)
os.makedirs("../data/results", exist_ok=True)
with open("../data/results/final_prompt_results.jsonl", "w") as f:
    for r in batch_results:
        f.write(json.dumps(r) + "\n")
print("Saved", len(batch_results), "results")


Key #1 rate-limited, trying next key...
Processed 10/121
Processed 20/121
Processed 30/121
Processed 40/121
Processed 50/121
Processed 60/121
Processed 70/121
Processed 80/121
Processed 90/121
Processed 100/121
Processed 110/121
FAILED at row 112: [Errno 54] Connection reset by peer
Processed 120/121
Saved 121 results


---
## 5. Phase 4 — Validation & Repair Loop

In [11]:
class ExtractionSchema(BaseModel):
    chief_complaint: Optional[str] = None
    symptoms: Optional[List[str]] = None
    diagnosis: Optional[str] = None
    medical_history: Optional[str] = None
    medications: Optional[List[str]] = None
    procedures: Optional[List[str]] = None
    follow_up: Optional[str] = None
    summary: str
    risk_indicators: Optional[str] = None
    urgency: Literal["low", "moderate", "high"]

def validate_and_repair(raw_json_text, schema_class, generate_fn, max_retries=2):
    for attempt in range(max_retries + 1):
        try:
            data = json.loads(raw_json_text)
            validated = schema_class(**data)
            return validated.model_dump(), True
        except Exception as e:
            if attempt == max_retries:
                return {"error": str(e), "raw_output": raw_json_text}, False
            raw_json_text = generate_fn(
                f"Your previous output failed validation with this error: {e}\n"
                f"Previous output: {raw_json_text}\n"
                f"Please fix it and return only valid JSON matching the required schema."
            ).text

def extract_final_validated(note_text):
    raw_output = extract_final(note_text)
    gen_fn = lambda p: generate_with_fallback(model=MODEL_NAME, contents=p)
    return validate_and_repair(raw_output, ExtractionSchema, gen_fn)


In [12]:
# Proof the repair loop actually triggers and fixes a genuine schema violation
def fake_generate_fn(prompt):
    print(">>> REPAIR TRIGGERED — asking model to fix its own output <<<")
    return generate_with_fallback(model=MODEL_NAME, contents=prompt)

broken_json = json.dumps({"chief_complaint": None, "symptoms": ["sad"], "summary": "test"})  # missing required "urgency"
fixed_data, ok = validate_and_repair(broken_json, ExtractionSchema, fake_generate_fn, max_retries=2)
print("\nSuccess after repair:", ok)
print(fixed_data)


>>> REPAIR TRIGGERED — asking model to fix its own output <<<
>>> REPAIR TRIGGERED — asking model to fix its own output <<<

Success after repair: True
{'chief_complaint': 'Not specified', 'symptoms': ['sad'], 'diagnosis': None, 'medical_history': None, 'medications': None, 'procedures': None, 'follow_up': None, 'summary': 'test', 'risk_indicators': None, 'urgency': 'low'}


---
## 6. Phase 3.5 — Live Medical Verification (Diagnosis / Medication / Dosing / ICD-10)

Every claim is checked against a real medical data source, never stated from the model's
memory alone. This version fixes two issues found in earlier testing:
1. A false-positive diagnosis on clearly positive/neutral text.
2. Confidence scores that never varied (always "low" regardless of evidence strength).

### Tools

In [13]:
import requests
import xml.etree.ElementTree as ET

def clean_html(text):
    if text is None:
        return None
    return re.sub(r"<[^>]+>", "", text).strip()

def condition_info_lookup(condition_text: str, max_results: int = 1) -> dict:
    """Look up authoritative medical info about a condition from MedlinePlus. Use this to verify a diagnosis before stating it."""
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {"db": "healthTopics", "term": condition_text, "rettype": "brief"}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    results = []
    for doc in root.findall(".//document")[:max_results]:
        title_el = doc.find(".//content[@name='title']")
        summary_el = doc.find(".//content[@name='snippet']")
        results.append({
            "title": clean_html(title_el.text if title_el is not None else None),
            "summary": clean_html(summary_el.text if summary_el is not None else None),
            "url": doc.attrib.get("url")
        })
    return {"found": bool(results), "results": results}


def medication_lookup(drug_name: str) -> dict:
    """Verify a medication name is real before suggesting it. If found is False, do not suggest that medication."""
    url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"
    params = {"name": drug_name}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    rxcui_list = data.get("idGroup", {}).get("rxnormId")
    if not rxcui_list:
        return {"found": False, "message": f"{drug_name!r} is not a recognized medication in RxNorm"}
    return {"found": True, "rxcui": rxcui_list[0], "name": drug_name}


def dosing_lookup(drug_name: str) -> dict:
    """Get standard FDA label dosing text for an already-verified medication. Generic reference only, never a personalized dose."""
    url = "https://api.fda.gov/drug/label.json"
    params = {"search": f'openfda.generic_name:"{drug_name}"', "limit": 1}
    response = requests.get(url, params=params, timeout=5)
    if response.status_code == 404:
        return {"found": False, "message": f"No FDA label found for {drug_name!r}"}
    response.raise_for_status()
    data = response.json()
    if not data.get("results"):
        return {"found": False, "message": f"No FDA label found for {drug_name!r}"}
    label = data["results"][0]
    dosage_text = label.get("dosage_and_administration", [None])[0]
    if not dosage_text:
        return {"found": True, "dosage_text": None, "note": "Label found but no dosage section available."}
    return {
        "found": True,
        "dosage_text": dosage_text[:500],
        "note": "Standard reference dose from FDA label — not patient-specific, for clinician review."
    }


def icd10_lookup(diagnosis_text: str, max_results: int = 3) -> dict:
    """Look up the official ICD-10-CM code for a confirmed diagnosis. Never state a code from memory."""
    url = "https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search"
    params = {"sf": "code,name", "terms": diagnosis_text, "maxList": max_results}
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    data = response.json()
    results = [{"code": c, "description": n} for c, n in data[3]]
    if not results:
        return {"found": False, "message": f"No ICD-10 code found for {diagnosis_text!r}"}
    return {"found": True, "codes": results}


In [14]:
# Sanity checks
print(condition_info_lookup("depression"))
print(medication_lookup("sertraline"))
print(medication_lookup("zorblatamine"))
print(dosing_lookup("sertraline"))
print(icd10_lookup("depression"))


{'found': True, 'results': [{'title': 'Depression', 'summary': "What is depression? Depression is more than a feeling of being sad or irritable for a few days. It's a serious mood disorder. As one of the most common  ...", 'url': 'https://medlineplus.gov/depression.html'}]}
{'found': True, 'rxcui': '36437', 'name': 'sertraline'}
{'found': False, 'message': "'zorblatamine' is not a recognized medication in RxNorm"}
{'found': True, 'dosage_text': 'DOSAGE AND ADMINISTRATION Initial Treatment Dosage for Adults Major Depressive Disorder –Sertraline hydrochloride treatment should be administered at a dose of 50 mg once daily. While a relationship between dose and effect has not been established for major depressive disorder, OCD, panic disorder, PTSD or social anxiety disorder, patients were dosed in a range of 50-200 mg/day in the clinical trials demonstrating the effectiveness of Sertraline hydrochloride for the treatment of this indication', 'note': 'Standard reference dose from FDA label

### MCP server exposing the tools

In [15]:
from mcp.server import MCPServer

mcp_server = MCPServer("medextract-tools")

@mcp_server.tool()
def condition_info_lookup_tool(condition_text: str) -> dict:
    """Look up authoritative medical info about a condition from MedlinePlus."""
    return condition_info_lookup(condition_text)

@mcp_server.tool()
def medication_lookup_tool(drug_name: str) -> dict:
    """Verify a medication name is real before suggesting it."""
    return medication_lookup(drug_name)

@mcp_server.tool()
def dosing_lookup_tool(drug_name: str) -> dict:
    """Get standard FDA label dosing for an already-verified medication."""
    return dosing_lookup(drug_name)

@mcp_server.tool()
def icd10_lookup_tool(diagnosis_text: str) -> dict:
    """Look up the official ICD-10-CM code for a confirmed diagnosis."""
    return icd10_lookup(diagnosis_text)


### Diagnosis reasoning — with the false-positive guard and confidence-calibration fix

In [24]:
class DiagnosisReasoningResult(BaseModel):
    diagnosis: Optional[str] = None
    icd10_code: Optional[str] = None
    icd10_description: Optional[str] = None
    suggested_medications: Optional[List[str]] = None
    unverified_medications_mentioned: Optional[List[str]] = None
    dosing_reference: Optional[str] = None
    confidence: Literal["low", "moderate", "high"]
    reasoning_notes: str

DIAGNOSIS_PROMPT = """You are reasoning about a possible diagnosis based on extracted symptoms or note content.

Rules:
- Before stating any diagnosis, call condition_info_lookup_tool to verify it against real medical information.
- Before suggesting any medication, call medication_lookup_tool to confirm it's a real medication.
  - If a mentioned medication is NOT found/verified, do NOT ask the user clarifying questions.
    Instead, put its name in "unverified_medications_mentioned" and do not include it in "suggested_medications".
- If you suggest a verified medication, call dosing_lookup_tool and copy its "dosage_text" value
  into "dosing_reference", followed by " (not patient-specific, for clinician review)".
  Do not paraphrase or invent the dosage — use the tool's dosage_text exactly.
- Once a diagnosis is confirmed via the tool, call icd10_lookup_tool to get its official code.
- Never state a diagnosis, medication, dose, or code without calling the matching tool first.

- GUARD AGAINST FALSE POSITIVES: if the input describes only positive, neutral, or coping content
  (e.g. happiness, hope, laughing with a friend, accomplishing tasks) with NO negative mood or
  symptom language, you MUST set "diagnosis", "icd10_code", "suggested_medications", and
  "dosing_reference" to null, "confidence" to "low", and explicitly state in reasoning_notes that
  no negative indicators were present.

- GUARD AGAINST OVER-SPECIFIC DIAGNOSES FROM VAGUE EVIDENCE: do not name a specific serious
  diagnosis (e.g. Bipolar Disorder, Generalized Anxiety Disorder, Adjustment Disorder) unless the
  text contains evidence specific to THAT diagnosis's actual defining features, not just any
  negative-sounding phrase. Specifically:
    - "Fluctuating moods" ALONE, with no mention of elevated mood, mania/hypomania symptoms
      (decreased need for sleep, grandiosity, racing thoughts, risky behavior), is NOT sufficient
      evidence for Bipolar Disorder. If this is the only evidence present, use a general descriptor
      like "Mood disturbance, unspecified" (or leave diagnosis null if confidence is very low) —
      do NOT name Bipolar Disorder or suggest lithium or any other specific mood-stabilizing
      medication from this phrase alone.
    - Do not suggest a medication with a narrow therapeutic window or significant monitoring
      requirements (e.g. lithium) unless the diagnosis it treats is itself well-supported by
      specific, multiple corroborating criteria in the text.
    - When evidence is generic/nonspecific, prefer a general descriptor and null/low-confidence
      medication suggestions over naming a specific severe condition just because some tool query
      happened to return one.
  State in reasoning_notes specifically which defining features of the named diagnosis were or
  were not present in the text.

- CONFIDENCE CALIBRATION: count the independent negative symptom/clinical indicators explicitly
  present in the text (e.g. sadness, low mood, isolation, anhedonia, fatigue, referral mentioned,
  "further assessment needed", explicit diagnosis word). Set confidence based on this count:
    - 0 indicators -> diagnosis must be null, confidence "low"
    - 1 indicator -> confidence "low"
    - 2 indicators -> confidence "moderate"
    - 3+ indicators, OR an explicit diagnosis word/clinical referral is present -> confidence "high"
  State the count and which indicators you found in reasoning_notes, so the calibration is
  auditable, not just asserted.
- Always respond with the structured output — never ask the user a clarifying question instead.
"""

def reason_diagnosis(input_text):
    response = generate_with_fallback(
        model=MODEL_NAME,
        contents=f"{DIAGNOSIS_PROMPT}\n\nInput: {input_text}",
        config={
            "tools": [condition_info_lookup, medication_lookup, dosing_lookup, icd10_lookup],
            "response_mime_type": "application/json",
            "response_schema": DiagnosisReasoningResult,
        }
    )
    return response.text


In [18]:
# Test 1: clear case (should be high-confidence, multiple indicators)
print("=== Test 1: clear multi-symptom case ===")
print(reason_diagnosis("feeling low, fatigue, loss of interest in activities, difficulty concentrating"))

# Test 2: fake medication (should flag as unverified, not silently drop or ask a question)
print("\n=== Test 2: fake medication ===")
print(reason_diagnosis("feeling low, fatigue, and I've been taking my usual zorbatrex for it"))

# Test 3: previously-failing false-positive case (should now correctly stay null)
print("\n=== Test 3: positive/neutral text (previously a false positive) ===")
print(reason_diagnosis("I felt hopeful today. I shared a laugh with a friend, which helped me momentarily."))

# Test 4: single weak indicator (should be low confidence, not automatically null, not high)
print("\n=== Test 4: single weak indicator ===")
print(reason_diagnosis("I felt a bit off today but managed to accomplish some tasks."))


=== Test 1: clear multi-symptom case ===


[09/13/26 21:44:46] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=551263;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=409149;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 21:45:00] INFO     HTTP Request: POST                                                     ]8;id=564634;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=314583;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 21:45:01] INFO     AFC remote call 1 is done.                                              ]8;id=206743;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=360526;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 21:45:17] INFO     HTTP Request: POST                                                     ]8;id=497072;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=144884;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

...

[09/13/26 21:47:17] INFO     AFC remote call 1 is done.                                              ]8;id=432004;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=339869;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 21:47:18] INFO     HTTP Request: POST                                                     ]8;id=833687;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=6752;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 21:47:19] INFO     AFC remote call 2 is done.                                              ]8;id=370393;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=472329;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 21:47:20] INFO     HTTP Request: POST                                                     ]8;id=116955;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=434648;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

{
  "diagnosis": null,
  "icd10_code": null,
  "icd10_description": null,
  "suggested_medications": null,
  "unverified_medications_mentioned": null,
  "dosing_reference": null,
  "confidence": "low",
  "reasoning_notes": "The input text describes feeling a bit off today but managing to accomplish some tasks. There are zero negative symptom/clinical indicators of a formal disorder present (accomplishing tasks and feeling only a bit off represents neutral/coping content with no significant negative mood or symptom language). Indicator count: 0. Per the guard against false positives and confidence calibration rules, diagnosis, icd10_code, suggested_medications, and dosing_reference are set to null, and confidence is low."
}


### Batch test across a stratified sample

In [19]:
sample_depression = eval_set[eval_set["gold_label"] == "depression"].sample(6, random_state=5)
sample_not = eval_set[eval_set["gold_label"] == "not depression"].sample(6, random_state=5)
sample_ambiguous = eval_set[eval_set["gold_label"] == "ambiguous"].sample(6, random_state=5)
phase35_sample = pd.concat([sample_depression, sample_not, sample_ambiguous]).reset_index(drop=True)
print(f"Sample size: {len(phase35_sample)}")

# Uncomment to actually run (each note may trigger multiple internal tool-call round-trips):
phase35_results = []
for i, row in phase35_sample.iterrows():
    try:
        raw = reason_diagnosis(row["text"])
        parsed = json.loads(raw)
        success = True
    except Exception as e:
        parsed = None; success = False
        print(f"FAILED at row {i}: {e}")
    phase35_results.append({"text": row["text"], "gold_label": row["gold_label"],
                             "model_output": parsed, "parse_success": success})
    time.sleep(5)
    if parsed:
        print(f"{i+1}/{len(phase35_sample)}: gold={row['gold_label']}, diagnosis={parsed.get('diagnosis')}, confidence={parsed.get('confidence')}")

os.makedirs("../data/results", exist_ok=True)
with open("../data/results/phase35_batch_results.jsonl", "w") as f:
    for r in phase35_results:
        f.write(json.dumps(r) + "\n")
print("Saved", len(phase35_results))


Sample size: 18


[09/13/26 21:47:39] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=132861;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=342864;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=714308;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=679616;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 21:47:41] INFO     AFC remote call 1 is done.                                              ]8;id=621447;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=750813;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=105517;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=498541;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

...

                    INFO     AFC remote call 3 is done.                                              ]8;id=166251;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=699133;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 21:51:31] INFO     HTTP Request: POST                                                     ]8;id=411717;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=883217;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 21:51:33] INFO     AFC remote call 4 is done.                                              ]8;id=445727;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=680371;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 21:51:34] INFO     HTTP Request: POST                                                     ]8;id=903417;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=991633;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

18/18: gold=ambiguous, diagnosis=Major Depressive Disorder, confidence=moderate
Saved 18


---
## 7. Phase 6 — Evaluation

In [20]:
def load_results(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def score_extraction(results):
    print(f"Total rows: {len(results)}")
    parse_success = sum(r["parse_success"] for r in results)
    print(f"Parse success rate: {parse_success}/{len(results)} = {parse_success/len(results):.1%}\n")

    for field in ["medications", "procedures", "medical_history"]:
        should_be_null = [r for r in results if r["gold_json"].get(field) is None]
        if not should_be_null:
            continue
        correctly_null = sum(1 for r in should_be_null if r["model_output"] and r["model_output"].get(field) is None)
        print(f"{field}: correctly null {correctly_null}/{len(should_be_null)} = {correctly_null/len(should_be_null):.1%}")

    from collections import Counter
    print("\nUrgency distribution:", Counter(r["model_output"].get("urgency") if r["model_output"] else None for r in results))
    missing_summary = sum(1 for r in results if not r["model_output"] or not r["model_output"].get("summary"))
    print(f"Rows missing summary: {missing_summary}/{len(results)}")

def score_reasoning(results):
    from collections import Counter
    success = [r for r in results if r.get("parse_success")]
    print(f"Parse success: {len(success)}/{len(results)}")
    print("Confidence distribution:", Counter(r["model_output"].get("confidence") for r in success if r["model_output"]))
    false_positives = [r for r in success if r["gold_label"] == "not depression" and r["model_output"].get("diagnosis")]
    print(f"False positives (diagnosed 'not depression' notes): {len(false_positives)}")
    for r in false_positives:
        print("  -", r["text"], "->", r["model_output"]["diagnosis"])

# Uncomment once the corresponding results files exist:
extraction_results = load_results("../data/results/final_prompt_results.jsonl")
score_extraction(extraction_results)

reasoning_results = load_results("../data/results/phase35_batch_results.jsonl")
score_reasoning(reasoning_results)


Total rows: 121
Parse success rate: 120/121 = 99.2%

medications: correctly null 120/121 = 99.2%
procedures: correctly null 120/121 = 99.2%
medical_history: correctly null 120/121 = 99.2%

Urgency distribution: Counter({'low': 106, 'moderate': 14, None: 1})
Rows missing summary: 1/121
Parse success: 17/18
Confidence distribution: Counter({'low': 11, 'high': 3, 'moderate': 3})
False positives (diagnosed 'not depression' notes): 1
  - Patient has fluctuating moods, feeling hopeful. -> Mood disorder, unspecified


-----------

In [21]:
def process_note(note_text: str) -> dict:
    """
    Full pipeline for one note:
      1. Extraction (Phase 3, FINAL prompt) -> validated against ExtractionSchema (Phase 4)
      2. Diagnosis/medication/dosing reasoning (Phase 3.5), verified live against real
         medical data sources.
    Returns a single combined dict with both results kept as separate sections, since they
    answer different questions: "extraction" is what's literally stated in the note;
    "diagnosis_reasoning" is a verified clinical suggestion for a doctor to review.
    """
    # --- Step 1: extraction ---
    raw_extraction = extract_final(note_text)
    gen_fn = lambda p: generate_with_fallback(model=MODEL_NAME, contents=p)
    extraction_result, extraction_ok = validate_and_repair(raw_extraction, ExtractionSchema, gen_fn)

    # --- Step 2: diagnosis / medication / dosing reasoning ---
    try:
        raw_reasoning = reason_diagnosis(note_text)
        reasoning_result = json.loads(raw_reasoning)
        reasoning_ok = True
    except Exception as e:
        reasoning_result = {"error": str(e)}
        reasoning_ok = False

    return {
        "note_text": note_text,
        "extraction": extraction_result,
        "extraction_valid": extraction_ok,
        "diagnosis_reasoning": reasoning_result,
        "reasoning_valid": reasoning_ok,
    }

In [22]:
import pprint

test_cases = [
    "Patient demonstrated symptoms of depression; further assessment needed.",
    "I felt hopeful today. I shared a laugh with a friend, which helped me momentarily.",
    "Patient feels low; a referral to a psychiatrist may be necessary.",
]

for note in test_cases:
    print("="*70)
    print("NOTE:", note)
    result = process_note(note)
    pprint.pprint(result)
    print()

NOTE: Patient demonstrated symptoms of depression; further assessment needed.


[09/13/26 22:00:20] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=483921;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=760509;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 22:00:22] INFO     HTTP Request: POST                                                     ]8;id=378024;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=67169;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=285986;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=305785;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 22:00:23] INFO     HTTP Request: POST                                                     ]8;id=897037;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=855249;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

...

[09/13/26 22:01:57] INFO     AFC remote call 3 is done.                                              ]8;id=404675;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=597542;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 22:01:58] INFO     HTTP Request: POST                                                     ]8;id=353941;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=173757;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

[09/13/26 22:02:00] INFO     AFC remote call 4 is done.                                              ]8;id=699507;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=49572;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 22:02:01] INFO     HTTP Request: POST                                                     ]8;id=427229;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=293636;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

{'diagnosis_reasoning': {'confidence': 'high',
                         'diagnosis': 'Major Depressive Disorder',
                         'dosing_reference': 'DOSAGE AND ADMINISTRATION '
                                             'Initial Treatment Dosage for '
                                             'Adults Major Depressive Disorder '
                                             '–Sertraline hydrochloride '
                                             'treatment should be administered '
                                             'at a dose of 50 mg once daily. '
                                             'While a relationship between '
                                             'dose and effect has not been '
                                             'established for major depressive '
                                             'disorder, OCD, panic disorder, '
                                             'PTSD or social anxiety disorder, '
                         

In [27]:
# Load already-completed rows so we don't redo them
completed_indices = set()
try:
    with open("../data/results/phase35_full_batch_progress.jsonl") as f:
        previous_results = [json.loads(line) for line in f]
    completed_indices = {r["index"] for r in previous_results}
    print(f"Found {len(completed_indices)} already-completed rows, resuming from there")
except FileNotFoundError:
    previous_results = []
    print("No previous progress found, starting from scratch")

remaining = eval_set[~eval_set.index.isin(completed_indices)]
print(f"Remaining to process: {len(remaining)}")

Found 40 already-completed rows, resuming from there
Remaining to process: 81


In [ ]:
def run_reasoning_batch(eval_df, reasoning_fn, delay=5, existing_results=None):
    results = list(existing_results) if existing_results else []
    for i, row in eval_df.iterrows():
        try:
            raw = reasoning_fn(row["text"])
            parsed = json.loads(raw)
            success = True
        except Exception as e:
            parsed = None
            success = False
            print(f"FAILED at row {i}: {e}")

        results.append({
            "index": i,
            "source": row["source"],
            "text": row["text"],
            "gold_label": row["gold_label"],
            "model_output": parsed,
            "parse_success": success
        })

        time.sleep(delay)
        if parsed:
            print(f"{i+1}: gold={row['gold_label']}, diagnosis={parsed.get('diagnosis')}, confidence={parsed.get('confidence')}")
        else:
            print(f"{i+1}: ERROR")

        if len(results) % 20 == 0:
            with open("../data/results/phase35_full_batch_progress.jsonl", "w") as f:
                for r in results:
                    f.write(json.dumps(r) + "\n")
            print(f"  -- progress saved ({len(results)} total rows) --")
    return results

phase35_full_results = run_reasoning_batch(remaining, reason_diagnosis, delay=5, existing_results=previous_results)

with open("../data/results/phase35_full_batch_results.jsonl", "w") as f:
    for r in phase35_full_results:
        f.write(json.dumps(r) + "\n")
print("\nFinal save complete:", len(phase35_full_results), "results")

In [29]:
import os

if os.path.exists("../data/results/phase35_full_batch_progress.jsonl"):
    os.remove("../data/results/phase35_full_batch_progress.jsonl")
    print("Old progress file deleted — starting fresh with the fixed prompt only")

phase35_full_results = run_reasoning_batch(eval_set, reason_diagnosis, delay=5, existing_results=[])

with open("../data/results/phase35_full_batch_results.jsonl", "w") as f:
    for r in phase35_full_results:
        f.write(json.dumps(r) + "\n")
print("\nFinal save complete:", len(phase35_full_results), "results")

Old progress file deleted — starting fresh with the fixed prompt only


[09/13/26 23:29:13] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=833524;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=391978;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

[09/13/26 23:29:14] INFO     HTTP Request: POST                                                     ]8;id=204952;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=357426;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

                    INFO     AFC remote call 1 is done.                                              ]8;id=483487;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=29057;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6289\6289]8;;\

[09/13/26 23:29:15] INFO     HTTP Request: POST                                                     ]8;id=466896;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=836890;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 200 OK"                                             

...

[09/13/26 23:45:10] INFO     HTTP Request: POST                                                     ]8;id=517520;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=847278;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 429 Too Many Requests"                              

Key #1 rate-limited, trying next key...


                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=107099;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=434714;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/google/genai/models.py#6243\6243]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=12511;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=817329;file:///opt/miniconda3/envs/deep-learning-course-2025/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh-lite:generateContent "HTTP/1.1 429 Too Many Requests"                              

Key #2 rate-limited, trying next key...
FAILED at row 120: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.5-flash-lite\nPlease retry in 49.261221391s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': 

In [30]:
import json, re
from collections import Counter

def load_results(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def _tokens(s):
    """Lowercase word tokens from a string, or from all strings in a list."""
    if isinstance(s, list):
        return _list_tokens(s)
    return set(re.findall(r"[a-z']+", s.lower())) if s else set()

def _list_tokens(lst):
    toks = set()
    for item in (lst or []):
        toks |= _tokens(item)
    return toks

def field_overlap_score(gold_val, model_val, is_list):
    """Token-overlap (recall) of model_val against gold_val for one field.
    Returns None when gold has no value for this field (not applicable to content accuracy)."""
    if gold_val is None:
        return None
    gold_toks = _list_tokens(gold_val) if is_list else _tokens(gold_val)
    if not gold_toks:
        return None
    if model_val is None:
        return 0.0
    model_toks = _list_tokens(model_val) if is_list else _tokens(model_val)
    return len(gold_toks & model_toks) / len(gold_toks)

def score_extraction(results):
    n = len(results)
    print(f"Total rows: {n}")
    parse_success = sum(r["parse_success"] for r in results)
    print(f"Parse success rate: {parse_success}/{n} = {parse_success/n:.1%}\n")

    parsed = [r for r in results if r["parse_success"] and r["model_output"]]

    # --- Measure 1: content-field accuracy, for fields that usually carry real values ---
    print("-- Content-field accuracy (token overlap vs. gold; symptoms, risk_indicators) --")
    for field, is_list in [("symptoms", True), ("risk_indicators", False)]:
        scores = [field_overlap_score(r["gold_json"].get(field), r["model_output"].get(field), is_list)
                  for r in parsed]
        scores = [s for s in scores if s is not None]
        if scores:
            avg = sum(scores) / len(scores)
            hit = sum(1 for s in scores if s > 0)
            print(f"  {field}: avg token overlap {avg:.1%} over {len(scores)} gold-populated rows "
                  f"(>=1 matching token in {hit}/{len(scores)})")

    # --- Measure 2: null-integrity, for fields that are usually absent ---
    print("\n-- Null-integrity check (fields that are usually null) --")
    for field in ["medications", "procedures", "medical_history", "follow_up"]:
        should_null = [r for r in parsed if r["gold_json"].get(field) is None]
        should_fill = [r for r in parsed if r["gold_json"].get(field) is not None]
        line = ""
        if should_null:
            ok = sum(1 for r in should_null if r["model_output"].get(field) is None)
            line += f"  {field}: correctly null {ok}/{len(should_null)} = {ok/len(should_null):.1%}"
        if should_fill:
            ok2 = sum(1 for r in should_fill if r["model_output"].get(field) is not None)
            line += f"  |  correctly non-null when gold has a value: {ok2}/{len(should_fill)} = {ok2/len(should_fill):.1%}"
        print(line)

    print("\nUrgency distribution (model):", Counter(r["model_output"].get("urgency") for r in parsed))
    missing_summary = sum(1 for r in parsed if not r["model_output"].get("summary"))
    print(f"Rows missing summary: {missing_summary}/{len(parsed)}")

extraction_results = load_results("../data/results/final_prompt_results.jsonl")
score_extraction(extraction_results)

Total rows: 121
Parse success rate: 120/121 = 99.2%

-- Content-field accuracy (token overlap vs. gold; symptoms, risk_indicators) --
  symptoms: avg token overlap 35.4% over 98 gold-populated rows (>=1 matching token in 68/98)
  risk_indicators: avg token overlap 6.7% over 15 gold-populated rows (>=1 matching token in 1/15)

-- Null-integrity check (fields that are usually null) --
  medications: correctly null 120/120 = 100.0%
  procedures: correctly null 120/120 = 100.0%
  medical_history: correctly null 120/120 = 100.0%
  follow_up: correctly null 103/103 = 100.0%  |  correctly non-null when gold has a value: 10/17 = 58.8%

Urgency distribution (model): Counter({'low': 106, 'moderate': 14})
Rows missing summary: 0/120


In [31]:
import json
from collections import Counter

def load_results(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def score_reasoning(results):
    success = [r for r in results if r.get("parse_success")]
    print(f"Parse success: {len(success)}/{len(results)}")
    print("Confidence distribution:", Counter(r["model_output"].get("confidence") for r in success if r["model_output"]))
    false_positives = [r for r in success if r["gold_label"] == "not depression" and r["model_output"].get("diagnosis")]
    print(f"False positives (diagnosed 'not depression' notes): {len(false_positives)}")
    for r in false_positives:
        print("  -", r["text"], "->", r["model_output"]["diagnosis"])

reasoning_results = load_results("../data/results/phase35_batch_results.jsonl")
score_reasoning(reasoning_results)

Parse success: 17/18
Confidence distribution: Counter({'low': 11, 'high': 3, 'moderate': 3})
False positives (diagnosed 'not depression' notes): 1
  - Patient has fluctuating moods, feeling hopeful. -> Mood disorder, unspecified


In [32]:
import json
from collections import Counter

def load_results(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def score_verification(results):
    """Phase 6, Measure 3: verification-pass rate — how often the diagnosis/medication/dosing
    suggestions actually pass their respective MCP tool checks, computed from the full-batch
    Phase 3.5 run across all 121 notes."""
    n = len(results)
    parsed = [r for r in results if r.get("parse_success") and r.get("model_output")]
    print(f"Parse success: {len(parsed)}/{n} = {len(parsed)/n:.1%}")
    if len(parsed) / n < 0.5:
        print("NOTE: parse-success rate is low on this full-batch run, most likely free-tier API "
              "rate-limiting across ~121 notes x multiple tool round-trips each, not a prompt/logic "
              "issue -- the same prompt hit 17/18 (94%) on the smaller stratified batch above. "
              "Reporting this honestly rather than hiding the failed rows.")
    print("\nConfidence distribution:", Counter(r["model_output"].get("confidence") for r in parsed))

    false_positives = [r for r in parsed if r["gold_label"] == "not depression" and r["model_output"].get("diagnosis")]
    not_dep_total = sum(1 for r in parsed if r["gold_label"] == "not depression")
    if not_dep_total:
        print(f"\nFalse positives (diagnosis asserted on 'not depression' notes): {len(false_positives)}/{not_dep_total}")
        for r in false_positives:
            print("  -", r["text"], "->", r["model_output"]["diagnosis"])

    # Diagnosis -> ICD-10: once the model confirms a diagnosis, did icd10_lookup resolve a code?
    diagnosed = [r for r in parsed if r["model_output"].get("diagnosis")]
    if diagnosed:
        icd10_ok = sum(1 for r in diagnosed if r["model_output"].get("icd10_code"))
        print(f"\nDiagnosis -> ICD-10 verification pass rate: {icd10_ok}/{len(diagnosed)} = {icd10_ok/len(diagnosed):.1%}")

    # Medication: of notes where the model engaged with a medication at all (suggested one, or
    # flagged one as unverified), what fraction had zero RxNorm rejects on this attempt?
    med_engaged = [r for r in parsed if r["model_output"].get("suggested_medications")
                   or r["model_output"].get("unverified_medications_mentioned")]
    if med_engaged:
        clean = sum(1 for r in med_engaged if not r["model_output"].get("unverified_medications_mentioned"))
        print(f"Medication verification pass rate (no RxNorm rejects): {clean}/{len(med_engaged)} = {clean/len(med_engaged):.1%}")
        rejected = len(med_engaged) - clean
        if rejected:
            print(f"  Rows with >=1 medication rejected by RxNorm verification: {rejected}")

    # Dosing: of notes with a verified suggested medication, did DailyMed return a usable dose?
    suggested = [r for r in parsed if r["model_output"].get("suggested_medications")]
    if suggested:
        dosed = sum(1 for r in suggested if r["model_output"].get("dosing_reference"))
        print(f"Dosing verification pass rate: {dosed}/{len(suggested)} = {dosed/len(suggested):.1%}")

full_reasoning_results = load_results("../data/results/phase35_full_batch_results.jsonl")
score_verification(full_reasoning_results)

Parse success: 21/121 = 17.4%
NOTE: parse-success rate is low on this full-batch run, most likely free-tier API rate-limiting across ~121 notes x multiple tool round-trips each, not a prompt/logic issue -- the same prompt hit 17/18 (94%) on the smaller stratified batch above. Reporting this honestly rather than hiding the failed rows.

Confidence distribution: Counter({'low': 12, 'high': 6, 'moderate': 3})

Diagnosis -> ICD-10 verification pass rate: 15/15 = 100.0%
Medication verification pass rate (no RxNorm rejects): 14/14 = 100.0%
Dosing verification pass rate: 14/14 = 100.0%
